# **Hill Country Revenue Forecast**: Regression — HF Deployment
---

## **Value Proposition**

In order to predict weekly product-store revenue across a regional Texas grocery chain, a gradient boosting regression model was developed that achieves R² = 0.8996 on the held-out test set, enabling merchandising teams to forecast per-item weekly revenue before placing orders and setting shelf allocations. Accurate weekly forecasts reduce over-ordering by flagging low-revenue items early and surface which promo configurations lift revenue, directly informing markdown and inventory decisions.

### Business Opportunities

* **Opportunity A:** Store managers currently rely on prior-week actuals and gut feel when setting shelf allocations and ordering quantities. Weekly revenue per item-store combination varies significantly across departments and store formats.
* **Solution A:** The model predicts `Weekly_Revenue_USD` per item × store combination one week ahead using pricing, packaging, and store attributes. It achieves a mean absolute error of $12.45 on the held-out test set, keeping prediction error within one standard deviation of typical weekly fluctuations for most product lines.

* **Opportunity B:** Promotion pricing (`Promo_Price_USD` vs `List_Price_USD`) is a direct lever the chain controls weekly. Without a predictive model, the revenue impact of a 10% discount is estimated from historical averages that don't account for item or store context.
* **Solution B:** The deployed `/predict` endpoint accepts any promo price as input, so category managers can compare predicted revenue at list price versus promo price for any item-store pair before committing to a markdown.

* **Opportunity C:** The forecast model is deployed as a live API on Hugging Face Spaces, with a Streamlit frontend for direct use by non-technical staff.
* **Solution C** (Options): Single-item point prediction with 95% confidence interval. Multi-item basket total across any store combination. Batch upload of CSV rows for bulk forecasting against the full product catalog.

### Outcomes

* **Model Performance**
  * HistGradientBoostingRegressor selected as the winning algorithm across a four-candidate sweep
  * Test R² = 0.8996, Test MAPE = 15.4%, Test MAE = $12.45
  * Boosting outperformed the ensemble methods (ExtraTreesRegressor: 0.8600, RandomForestRegressor: 0.8454) on the validation leaderboard, consistent with the structured tabular features in this dataset

* **Architecture**
  * Gated pipeline harness sweeps RandomForest, ExtraTrees, HistGradientBoosting, XGBoost; winner selected by validation R² — validated by [Notebook A (leak-detection)](https://evagaiml.github.io/000-gated-pipeline-cases/leak-detection__regression__gated-harness/)
  * Split-conformal prediction intervals: 1,776-row calibration set, calibrated q_low = -36.14, q_high = 45.83

* **Economic Impact**
  * A MAE of $12.45/week per item-store row translates to ordering decisions made within roughly one standard error of the actual week's revenue for most SKUs
  * The promo lever in the prediction form quantifies discount ROI before a markdown decision is made, avoiding revenue dilution from unnecessary promos on items that would have sold at list price

* **Strategy Recommendation**
  * **Enterprise:** Integrate `/predict_batch` into the weekly replenishment workflow via API call from the ERP, with model retrain on a rolling 52-week window as new sales data accumulates
  * **SMB:** Use the Streamlit UI for weekly planning meetings; category managers input the upcoming week's planned promotions and read the predicted revenue impact per item before finalizing the flyer

## **Problem Space**

### Overview

* Weekly revenue per item-store combination drives ordering quantities, shelf allocation, and promo calendar decisions across the Hill Country Grocer chain. Getting it wrong in either direction costs the chain: over-forecast leads to excess inventory and markdowns; under-forecast leads to stockouts and lost sales.
* The Hill Country Grocer chain spans 6 stores across three Texas regions (South Texas Central, Gulf, and Hill Country). Revenue varies by department, store size, and promo status — a model that ignores any of these axes will produce systematically biased forecasts for the stores or categories it under-represents.
* `Weekly_Revenue_USD` is the product of units sold and effective price. The dataset does not carry a temporal anchor (no week-of-year column), so seasonal trends are not directly modeled here. That limitation is surfaced explicitly in Section 4.

### Data Description

* **8,880** weekly item × store observations total: **5,328** training rows, **1,776** calibration rows, and **1,776** test rows
* 12 predictive features after dropping the display-only `Item_Description` column and the high-cardinality `UPC` identifier: 5 categorical (Department, Brand_Type, Store_Number, Store_Banner, Store_Region) and 7 numeric (Net_Weight_oz, Pack_Size, List_Price_USD, Promo_Price_USD, Shelf_Facings, Store_Sq_Ft, Store_Open_Year)
* Data source: `hill_country_grocer_weekly_sales.csv`, loaded via the shared Layer-2 data loader
* Target variable: `Weekly_Revenue_USD` — continuous, min $0.87, no zeros
* Secondary target: `Weekly_Units_Sold` — excluded from primary model as a co-target per the gated pipeline's leakage rules; the model predicts revenue directly

### Process

* Three-way split (60% / 20% / 20%) enforced before any model fitting; the 20% calibration set is held out entirely for split-conformal CI computation and never seen during model selection or tuning.
* Gated pipeline harness sweeps four candidates (RandomForest, ExtraTrees, HistGradientBoosting, XGBoost) on an internal validation split of the training set; the validation leaderboard winner proceeds to GridSearchCV tuning.
* Conformal quantiles (q_low = -36.14, q_high = 45.83) computed on the held-out calibration set using signed residuals; these scalars shift every prediction to produce the 95% interval, making the CI mechanism model-agnostic.

### Results
>| Model | Val R² | Role |
>|---|---|---|
>| **HistGradientBoostingRegressor ✅** | 0.8906 | Lead deployment model |
>| XGBRegressor | 0.8869 | Candidate |
>| ExtraTreesRegressor | 0.8600 | Candidate |
>| RandomForestRegressor | 0.8454 | Candidate |

# **Code Execution**

### **Runtime Configuration**

> **Hardware Accelerator:** **CPU** and **Standard RAM**
>
> This notebook trains a gradient boosting model on tabular data and deploys to Hugging Face Spaces. No GPU is required.

*   Installing Colab Dependencies may require a Runtime restart. You will be notified if a restart is required.
*   HF_TOKEN must be set in Colab Secrets (key icon → +Add new secret → name=HF_TOKEN, Notebook access ON) before running the deployment cells.

In [ ]:
# ------------------------------
# INSTALL DEPENDENCIES
# ------------------------------
# Installs packages required for training, data access, and HF deployment.
# Uses a flag file to skip re-install after runtime restart.
import os, sys
from pathlib import Path
from IPython.display import display, HTML

# Use /content flag in Colab; local .deps_installed_flag otherwise
_in_colab = os.path.exists("/content")
flag_file = Path("/content/.deps_installed_flag") if _in_colab else Path(".deps_installed_flag")

if not flag_file.exists():
    print("Installing dependencies... This may take a minute or two.")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "scikit-learn==1.6.1", "xgboost>=1.7",
        "joblib", "pandas", "numpy", "pyarrow",
        "huggingface_hub>=0.25", "requests",
    ], check=True)
    flag_file.touch()
    if _in_colab:
        display(HTML("""
        <div style="
            border: 4px solid #d93025;
            background: #fce8e6;
            color: #202124;
            padding: 24px;
            border-radius: 12px;
            font-family: Arial, sans-serif;
            margin: 16px 0;
        ">
            <div style="font-size: 30px; font-weight: 800; color: #b31412; margin-bottom: 12px;">
                ⚠️ Runtime Restart Required
            </div>
            <div style="font-size: 22px; line-height: 1.5; font-weight: 600;">
                Dependencies have been successfully installed!
                <br><br>
                Please go to <b>Runtime → Restart session and run all</b> in the top menu.
            </div>
        </div>
        """))
    else:
        print("Dependencies installed.")
else:
    print("✅ Dependencies already installed. No restart required.")

### Library Import and Configuration

**Process:** Import all required libraries and set the random seed for reproducible splits and model training.

**Outcome:** Environment is configured; all dependencies are confirmed importable.

In [ ]:
# ------------------------------
# LIBRARY IMPORT AND CONFIGURATION
# ------------------------------
# Standard data science libraries, sklearn, XGBoost, joblib, and HF hub.
import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

# Data handling
import numpy as np
import pandas as pd
import joblib
import pyarrow

# Sklearn: model selection, preprocessing, metrics
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import r2_score, mean_absolute_percentage_error, mean_absolute_error
from sklearn.inspection import permutation_importance

# XGBoost
from xgboost import XGBRegressor

# HF deployment
from huggingface_hub import HfApi

# Reproducibility
SEED = 42
TARGET = "Weekly_Revenue_USD"
CO_TARGET = "Weekly_Units_Sold"
DISPLAY_ONLY = ["Item_Description"]

print("sklearn:", __import__("sklearn").__version__)
print("xgboost:", __import__("xgboost").__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

### HF Authentication

**Process:** Verify the `HF_TOKEN` Colab secret and confirm the authenticated HF username. The token must be set in Colab Secrets before running this cell.

**Outcome:** `api` object is ready and `HF_OWNER` resolves to the authenticated user's HF namespace.

In [ ]:
# ------------------------------
# HF AUTHENTICATION
# ------------------------------
# Reads HF_TOKEN from Colab Secrets or environment; never prompts for input.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass   # not in Colab — HF_TOKEN may already be in env

if not os.environ.get("HF_TOKEN"):
    raise EnvironmentError(
        "HF_TOKEN not set. Add it to Colab Secrets (key icon → +Add new secret "
        "→ name=HF_TOKEN, value=<your token>, Notebook access ON) and re-run this cell."
    )

api = HfApi()
me = api.whoami()
HF_OWNER = me["name"]
print(f"Authenticated as: {HF_OWNER}")
BACKEND_REPO = f"{HF_OWNER}/hill-country-revenue-forecast-backend"
FRONTEND_REPO = f"{HF_OWNER}/hill-country-revenue-forecast-frontend"
BACKEND_URL = f"https://{HF_OWNER.lower()}-hill-country-revenue-forecast-backend.hf.space"
print(f"Backend:  {BACKEND_REPO}")
print(f"Frontend: {FRONTEND_REPO}")

### Data Loading

**Process:** Load `hill_country_grocer_weekly_sales.csv` via the shared Layer-2 data loader, which downloads and SHA-256 verifies the dataset zip from the `000-gated-pipeline-cases-data` repository.

**Analysis:** The dataset is 8,880 weekly item × store observations. Rows are independent snapshots; there is no temporal column, so no train/val leakage via time ordering.

**Outcome:** `df` contains all 8,880 rows across 16 columns. The target `Weekly_Revenue_USD` has no zeros, making MAPE a valid evaluation metric.

In [ ]:
# ------------------------------
# DATA LOADING
# ------------------------------
# Uses the shared data loader to download and verify the hill country dataset.
# Works identically in Colab (/content/data_cache/) and locally (./data_cache/).
import importlib.util, pathlib, sys

# Import the shared data_access module (handles Colab vs local path detection)
_shared_path = pathlib.Path("_shared")
if not _shared_path.exists():
    _shared_path = pathlib.Path("/content/notebooks/_shared")
spec = importlib.util.spec_from_file_location("data_access", _shared_path / "data_access.py")
data_access = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_access)

data_dir = data_access.ensure_dataset("hill-country-revenue-forecast__regression__hf-deployment")
csv_path = data_dir / "hill_country_grocer_weekly_sales.csv"
df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nTarget stats:")
print(df[TARGET].describe().round(2))

### Data Overview and Feature Selection

**Process:** Profile the dataset, identify identifier columns by the harness rule (unique-value ratio > 0.9), and define the final feature set. Drop `Item_Description` (display label, not predictive) and `Weekly_Units_Sold` (co-target — predicting revenue from units sold is circular).

**Analysis:** UPC is flagged as a high-cardinality identifier (unique ratio > 0.9) and dropped. The remaining 12 features span item attributes, store attributes, and pricing — each an independent predictor of weekly revenue.

**Outcome:** `X` has 12 features (5 categorical, 7 numeric). Target `y` is `Weekly_Revenue_USD`.

In [ ]:
# ------------------------------
# DATA OVERVIEW AND FEATURE SELECTION
# ------------------------------
# Drop identifier columns, co-target, and display-only text column.
# Categorical / numeric split drives OHE preprocessing.

# Co-target and display-only drops (spec-mandated)
y = df[TARGET].copy()
X = df.drop(columns=[TARGET, CO_TARGET] + DISPLAY_ONLY)

# Identifier drop: same rule as gated_pipeline.py (nunique/len > 0.9)
id_cols = [c for c in X.columns if X[c].nunique() / len(X) > 0.9]
print(f"Identifier columns dropped (nunique/len > 0.9): {id_cols}")
if id_cols:
    X = X.drop(columns=id_cols)

cats = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
nums = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
print(f"\nFinal feature set ({len(X.columns)} features):")
print(f"  Categorical ({len(cats)}): {cats}")
print(f"  Numeric ({len(nums)}): {nums}")
print(f"\nTarget range: ${y.min():.2f} – ${y.max():.2f}")
print(f"Target zeros: {(y == 0).sum()}")

### Exploratory Data Analysis

**Process:** Compute per-department revenue share and per-store row counts to characterize the dataset composition before splitting.

**Analysis:** Revenue concentration by department and store informs which categorical features carry the most signal. Unequal store sizes confirm Store_Number is a meaningful predictor, not just a grouping label.

**Outcome:** Department revenue shares and store row counts printed for reference.

In [ ]:
# ------------------------------
# EXPLORATORY DATA ANALYSIS
# ------------------------------
# Department revenue share and store distribution.

dept_rev = df.groupby("Department")[TARGET].sum().sort_values(ascending=False)
dept_share = (dept_rev / dept_rev.sum() * 100).round(1)
print("Department revenue share (%):")
print(dept_share.to_string())

print("\nRows per store:")
print(df["Store_Number"].value_counts().sort_index().to_string())

print("\nBrand_Type distribution:")
print(df["Brand_Type"].value_counts().to_string())

### Three-Way Split for Split-Conformal Prediction

**Process:** Partition the dataset into three non-overlapping sets before any model fitting: 60% training, 20% calibration, 20% test. The calibration set is held out entirely from model selection and tuning — it is only used after the winner is chosen to compute the conformal quantiles q_low and q_high.

**Analysis:** Split-conformal prediction requires a held-out calibration set that is exchangeable with the test set but was not used to train or select the model. Mixing calibration and training data would invalidate the coverage guarantee. The 60/20/20 partition gives 5,328 training rows (enough for robust tree fitting with cross-validation), 1,776 calibration rows (enough for stable 2.5th/97.5th percentile estimates), and 1,776 test rows.

**Outcome:** `X_train` (5,328), `X_cal` (1,776), `X_test` (1,776). OHE fit on `X_train` only.

In [ ]:
# ------------------------------
# THREE-WAY SPLIT AND PREPROCESSING
# ------------------------------
# Mandatory for split-conformal CI: calibration set held out before any model fitting.
# OHE encoder fit on training rows only — no test/cal information in the transform.

# Three-way split: 60% train, 20% calibration, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
X_train, X_cal, y_train, y_cal = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED   # 0.25 * 0.80 = 0.20 of total
)
print(f"Train: {len(X_train)} rows | Cal: {len(X_cal)} rows | Test: {len(X_test)} rows")

# One-hot encoding: min_frequency=10 (rare categories become 'infrequent_sklearn')
# max_categories=20 (caps high-cardinality categoricals)
ohe = ColumnTransformer(
    [("c", OneHotEncoder(handle_unknown="ignore", min_frequency=10,
                         max_categories=20, sparse_output=False), cats),
     ("n", "passthrough", nums)]
)
ohe.fit(X_train)

X_train_t = ohe.transform(X_train)
X_cal_t   = ohe.transform(X_cal)
X_test_t  = ohe.transform(X_test)
feature_names = ohe.get_feature_names_out().tolist()
print(f"OHE output dimension: {len(feature_names)} features")

### Model Selection — Gated Pipeline Sweep

**Process:** Evaluate four candidates — RandomForest, ExtraTrees, HistGradientBoosting, XGBoost — on a 20% internal validation split of the training set. The highest-validation-R² model proceeds to hyperparameter tuning.

**Analysis:** HistGradientBoostingRegressor led the leaderboard at val R² = 0.8906, with XGBoost close behind at 0.8869. Both boosting methods outperformed the bagging ensembles on this structured tabular dataset, consistent with gradient boosting's advantage when additive residuals are meaningful.

**Outcome:** `winner_name = 'HistGradientBoostingRegressor'`. Proceeds to GridSearchCV tuning.

In [ ]:
# ------------------------------
# MODEL SELECTION — GATED PIPELINE SWEEP
# ------------------------------
# Replicates the gated_pipeline.py candidate sweep on the training set.
# Validation is on an internal 20% hold-out of X_train (not the conformal cal set).

X_tr_inner, X_va_inner, y_tr_inner, y_va_inner = train_test_split(
    X_train_t, y_train, test_size=0.20, random_state=SEED
)

candidates = {
    "RandomForestRegressor":          RandomForestRegressor(n_estimators=80, random_state=SEED, n_jobs=-1),
    "ExtraTreesRegressor":            ExtraTreesRegressor(n_estimators=80, random_state=SEED, n_jobs=-1),
    "HistGradientBoostingRegressor":  HistGradientBoostingRegressor(random_state=SEED),
    "XGBRegressor":                   XGBRegressor(n_estimators=150, random_state=SEED, n_jobs=-1, verbosity=0),
}

leaderboard = []
for name, model in candidates.items():
    model.fit(X_tr_inner, y_tr_inner)
    r2_val = r2_score(y_va_inner, model.predict(X_va_inner))
    leaderboard.append((name, r2_val))
    print(f"  {name}: {r2_val:.4f}")

leaderboard.sort(key=lambda t: -t[1])
winner_name = leaderboard[0][0]
print(f"\nWinner: {winner_name}")

### Hyperparameter Tuning

**Process:** Run 3-fold GridSearchCV over `max_depth ∈ [4, 6, 8]` for HistGradientBoostingRegressor on the full training set. CV scoring is R². Best parameters selected by mean validation fold R².

**Analysis:** `max_depth = 8` maximized CV R² at 0.8852. Deeper trees capture more feature interactions but the best depth here was at the upper end of the search range, suggesting the interaction structure in this dataset benefits from trees that can express 3–4 feature combinations.

**Outcome:** `winner_model` is the best estimator from GridSearchCV, refitted on the full training set.

In [ ]:
# ------------------------------
# HYPERPARAMETER TUNING
# ------------------------------
# GridSearchCV over max_depth for the winning model family.
# 3-fold CV on full training set; best estimator refitted automatically.

tune_registry = {
    "RandomForestRegressor": (
        RandomForestRegressor(random_state=SEED, n_jobs=-1, n_estimators=120),
        {"max_depth": [6, 9, 12]}
    ),
    "ExtraTreesRegressor": (
        ExtraTreesRegressor(random_state=SEED, n_jobs=-1, n_estimators=120),
        {"max_depth": [6, 9, 12]}
    ),
    "HistGradientBoostingRegressor": (
        HistGradientBoostingRegressor(random_state=SEED, max_iter=200),
        {"max_depth": [4, 6, 8]}
    ),
    "XGBRegressor": (
        XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0, n_estimators=200),
        {"max_depth": [3, 5, 7]}
    ),
}

base_est, param_grid = tune_registry[winner_name]
gs = GridSearchCV(base_est, param_grid, cv=KFold(3, shuffle=True, random_state=SEED),
                  scoring="r2", n_jobs=-1)
gs.fit(X_train_t, y_train)
winner_model = gs.best_estimator_
print(f"Best params:  {gs.best_params_}")
print(f"CV R²:        {gs.best_score_:.4f}")

### Final Evaluation on Held-Out Test Set

**Process:** Predict on `X_test` (the 20% held-out test set, untouched until this step) and compute R², MAPE, and MAE.

**Analysis:** Test R² = 0.8996 confirms the model generalizes to unseen item-store combinations. MAPE = 15.4% reflects the wide revenue range in the dataset — items like produce sell for under $5/unit while household and meat categories can run $15–$20, so small-revenue rows inflate MAPE even when absolute error is low. MAE = $12.45 per item-store-week is the operationally relevant error metric.

**Outcome:** Model clears the Tier 2 threshold (R² ≥ 0.80) and is cleared for deployment.

In [ ]:
# ------------------------------
# FINAL EVALUATION — HELD-OUT TEST SET
# ------------------------------
# First and only prediction on the test set; produces the authoritative metrics.

y_test_pred = winner_model.predict(X_test_t)
test_r2   = float(r2_score(y_test, y_test_pred))
test_mape = float(mean_absolute_percentage_error(y_test, y_test_pred) * 100)
test_mae  = float(mean_absolute_error(y_test, y_test_pred))

print(f"Test R²:   {test_r2:.4f}")
print(f"Test MAPE: {test_mape:.2f}%")
print(f"Test MAE:  ${test_mae:.2f}")

### Feature Importances — Permutation Method

**Process:** Compute permutation importance on the test set (5 repeats). HistGradientBoostingRegressor does not expose tree-level MDI importances directly, so the model-agnostic permutation method is used. Each feature is randomly shuffled 5 times; importance is the mean drop in R² across permutations.

**Analysis:** `List_Price_USD` is the strongest single predictor (importance 1.35), followed by `Promo_Price_USD` (0.26). Pricing dominates, consistent with revenue being the product of unit price × units sold. Store location effects (`Store_Region_South - Texas Hill Country`) appear in the top 5, reflecting demand differences across the Texas regions.

**Outcome:** `top5_fi` list saved into the model bundle for display in `/info` and the frontend Insights section.

In [ ]:
# ------------------------------
# FEATURE IMPORTANCES — PERMUTATION
# ------------------------------
# Model-agnostic; works for any winner from the harness sweep.
# Permutation importance on the held-out test set.

perm = permutation_importance(winner_model, X_test_t, y_test,
                               n_repeats=5, random_state=SEED, n_jobs=-1)
fi = sorted(zip(feature_names, perm.importances_mean), key=lambda t: -t[1])
top5_fi = [(str(f), round(float(i), 4)) for f, i in fi[:5]]

print("Top 10 feature importances (permutation, test set):")
for fname, imp in fi[:10]:
    print(f"  {fname}: {imp:.4f}")

### Split-Conformal Calibration

**Process:** Predict on the held-out calibration set (`X_cal`, 1,776 rows) using the final fitted model. Compute signed residuals `y_cal − ŷ_cal`. The 2.5th and 97.5th percentiles of these residuals are the conformal quantiles q_low and q_high.

**Analysis:** q_low = -36.14 and q_high = 45.83 reflect an asymmetric distribution of residuals — the model under-predicts high-revenue weeks more than it over-predicts, which is expected for skewed revenue distributions. At inference, `ci_low = prediction + q_low` and `ci_high = prediction + q_high` shifts every point prediction to the 95% marginal coverage interval calibrated on this specific model and dataset.

**Outcome:** Quantiles saved into the model bundle. Empirical coverage on the test set is verified in the deployment cell.

In [ ]:
# ------------------------------
# SPLIT-CONFORMAL CALIBRATION
# ------------------------------
# Calibration set is exchangeable with test set and was never used for training.
# Signed residuals → 2.5th / 97.5th percentile quantiles.

y_cal_pred = winner_model.predict(X_cal_t)
residuals  = np.array(y_cal) - y_cal_pred    # signed: positive = model under-predicted
alpha      = 0.05
q_low  = float(np.quantile(residuals, alpha / 2))        # ~2.5th percentile
q_high = float(np.quantile(residuals, 1 - alpha / 2))    # ~97.5th percentile

print(f"Calibration set n:  {len(X_cal)}")
print(f"q_low  (2.5th pct): {q_low:.4f}")
print(f"q_high (97.5th pct): {q_high:.4f}")

# Empirical coverage on the TEST set (sanity check before deploying)
ci_low_test  = y_test_pred + q_low
ci_high_test = y_test_pred + q_high
covered = ((np.array(y_test) >= ci_low_test) & (np.array(y_test) <= ci_high_test)).mean()
print(f"\nEmpirical coverage on test set: {covered:.3f} (target: ~0.95)")

### Save Model Bundle

**Process:** Package the trained model, OHE transformer, feature metadata, test metrics, and conformal quantiles into a single `model.joblib` file. Also export a 1,000-row stratified training sample as `training_sample.parquet` for the frontend Insights endpoint.

**Outcome:** `model.joblib` and `training_sample.parquet` written to the current working directory, ready for upload to the HF backend Space.

In [ ]:
# ------------------------------
# SAVE MODEL BUNDLE
# ------------------------------
# Single bundle file for backend loading at startup.

bundle = {
    "model":                  winner_model,
    "ohe":                    ohe,
    "feature_names":          feature_names,
    "cats":                   cats,
    "nums":                   nums,
    "sklearn_version":        __import__("sklearn").__version__,
    "winner_name":            winner_name,
    "test_r2":                test_r2,
    "test_mape":              test_mape,
    "test_mae":               float(test_mae),
    "conformal_alpha":        alpha,
    "conformal_q_low":        q_low,
    "conformal_q_high":       q_high,
    "conformal_calibration_n": int(len(X_cal)),
    "train_n":                int(len(X_train)),
    "cal_n":                  int(len(X_cal)),
    "test_n":                 int(len(X_test)),
    "top5_fi":                top5_fi,
    "top_feature":            top5_fi[0][0] if top5_fi else "List_Price_USD",
    "top_feature_importance": top5_fi[0][1] if top5_fi else 0.0,
}
import os as _os, shutil as _shutil

# Determine artifact directory: /content in Colab, /tmp otherwise
_artifact_dir = "/content" if _os.path.exists("/content") else "/tmp"

joblib.dump(bundle, "model.joblib")
_shutil.copy("model.joblib", f"{_artifact_dir}/model.joblib")
print(f"Saved model.joblib → {_artifact_dir}/model.joblib")

# Training sample for frontend Insights
sample_df = X_train.sample(min(1000, len(X_train)), random_state=SEED)
sample_df.to_parquet("training_sample.parquet", index=False)
_shutil.copy("training_sample.parquet", f"{_artifact_dir}/training_sample.parquet")
print(f"Saved training_sample.parquet ({len(sample_df)} rows)")

import hashlib
sha = hashlib.sha256(open("model.joblib", "rb").read()).hexdigest()
size_bytes = _os.path.getsize("model.joblib")
print(f"model.joblib SHA-256: {sha}")
print(f"model.joblib size:    {size_bytes:,} bytes")

### Write Space Files

**Process:** Write the backend (FastAPI `app.py`, `Dockerfile`, `requirements.txt`) and frontend (Streamlit `streamlit_app.py`, `Dockerfile`, `requirements.txt`) files to the Colab filesystem for upload.

**Outcome:** All Space files written under `/content/hf-backend/` and `/content/hf-frontend/`.

In [ ]:
# ------------------------------
# WRITE HF SPACE FILES
# ------------------------------
# Backend and frontend Space file contents written to /content/ (Colab) or /tmp/ (local).
import pathlib, textwrap, os

_is_colab = os.path.exists("/content")
_base = pathlib.Path("/content") if _is_colab else pathlib.Path("/tmp")

# Backend
backend_dir = _base / "hf-backend"
backend_dir.mkdir(exist_ok=True)

(backend_dir / "Dockerfile").write_text(
    "FROM python:3.12-slim\n"
    "WORKDIR /app\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY app.py .\n"
    "COPY model.joblib .\n"
    "COPY training_sample.parquet .\n"
    "EXPOSE 7860\n"
    "CMD [\"uvicorn\", \"app:app\", \"--host\", \"0.0.0.0\", \"--port\", \"7860\"]\n"
)

(backend_dir / "requirements.txt").write_text(
    "fastapi==0.115.5\n"
    "uvicorn[standard]==0.32.0\n"
    "joblib==1.5.3\n"
    "scikit-learn==1.6.1\n"
    "pandas==2.2.3\n"
    "numpy==2.0.2\n"
    "pyarrow==17.0.0\n"
)

backend_app = '''"""Hill Country Grocer — Weekly Revenue Forecast API (v3.2, split-conformal CI)."""
from typing import Any
import joblib
import pandas as pd
import pyarrow.parquet as pq
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Hill Country Revenue Forecast API", version="3.2.0")

bundle = joblib.load("model.joblib")
MODEL = bundle["model"]
OHE = bundle["ohe"]
CATS = bundle["cats"]
NUMS = bundle["nums"]
FEATURE_NAMES = bundle["feature_names"]
WINNER_NAME = bundle["winner_name"]
TEST_R2 = bundle["test_r2"]
TEST_MAPE = bundle["test_mape"]
SKLEARN_VERSION = bundle["sklearn_version"]
Q_LOW = bundle["conformal_q_low"]
Q_HIGH = bundle["conformal_q_high"]
CAL_N = bundle["conformal_calibration_n"]
TOP5_FI = bundle.get("top5_fi", [])

_SAMPLE_DF = pq.read_table("training_sample.parquet").to_pandas()


def _prepare_df(raw):
    row = {col: raw.get(col, "Unknown" if col in CATS else 0.0) for col in CATS + NUMS}
    return pd.DataFrame([row])[CATS + NUMS]


class PredictRequest(BaseModel):
    features: dict[str, Any]

class BatchRequest(BaseModel):
    records: list[dict[str, Any]]


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/info")
def info():
    features = []
    ohe_t = OHE.named_transformers_["c"]
    for idx, col in enumerate(CATS):
        features.append({"name": col, "type": "categorical",
                          "allowed_values": [str(v) for v in ohe_t.categories_[idx]]})
    for col in NUMS:
        col_data = _SAMPLE_DF[col] if col in _SAMPLE_DF.columns else None
        entry = {"name": col, "type": "numeric"}
        if col_data is not None:
            entry["min"] = round(float(col_data.min()), 4)
            entry["max"] = round(float(col_data.max()), 4)
        features.append(entry)
    return {
        "model_family": WINNER_NAME,
        "target": "Weekly_Revenue_USD",
        "secondary_target": "Weekly_Units_Sold",
        "sklearn_version": SKLEARN_VERSION,
        "test_r2": round(TEST_R2, 4),
        "test_mape_pct": round(TEST_MAPE, 2),
        "ci_mechanism": "split_conformal",
        "ci_alpha": 0.05,
        "ci_coverage_pct": 95,
        "ci_calibration_n": CAL_N,
        "features": features,
        "top5_feature_importances": TOP5_FI,
    }


@app.post("/predict")
def predict(req: PredictRequest):
    try:
        df = _prepare_df(req.features)
        pred = float(MODEL.predict(OHE.transform(df))[0])
        return {"prediction": round(pred, 2),
                "ci_low": round(pred + Q_LOW, 2),
                "ci_high": round(pred + Q_HIGH, 2),
                "currency": "USD"}
    except Exception as e:
        raise HTTPException(status_code=422, detail=str(e))


@app.post("/predict_batch")
def predict_batch(req: BatchRequest):
    if not req.records:
        raise HTTPException(status_code=400, detail="records cannot be empty")
    try:
        df = pd.concat([_prepare_df(r) for r in req.records], ignore_index=True)
        preds = MODEL.predict(OHE.transform(df))
        results = [{"prediction": round(float(p), 2),
                    "ci_low": round(float(p) + Q_LOW, 2),
                    "ci_high": round(float(p) + Q_HIGH, 2)} for p in preds]
        return {"predictions": results, "count": len(results)}
    except Exception as e:
        raise HTTPException(status_code=422, detail=str(e))


@app.get("/training_sample")
def training_sample():
    return {"sample": _SAMPLE_DF.to_dict(orient="records"), "n": len(_SAMPLE_DF)}
'''
(backend_dir / "app.py").write_text(backend_app)

# Frontend
frontend_dir = _base / "hf-frontend"
(frontend_dir / "src").mkdir(parents=True, exist_ok=True)

(frontend_dir / "Dockerfile").write_text(
    "FROM python:3.12-slim\n"
    "WORKDIR /app\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY src/ src/\n"
    "EXPOSE 7860\n"
    'CMD ["streamlit", "run", "src/streamlit_app.py", "--server.port", "7860", "--server.address", "0.0.0.0"]\n'
)
(frontend_dir / "requirements.txt").write_text(
    "streamlit==1.40.1\n"
    "requests==2.32.3\n"
    "pandas==2.2.3\n"
    "pyarrow==17.0.0\n"
)

print(f"Space files written to {_base}")
print("  Backend:", [f.name for f in backend_dir.iterdir()])
print("  Frontend src:", [f.name for f in (frontend_dir / "src").iterdir()])


### Write Streamlit Frontend App

**Process:** Write the full Streamlit frontend source to `/content/hf-frontend/src/streamlit_app.py`.

**Outcome:** Frontend source file ready for upload.

In [ ]:
# ------------------------------
# WRITE STREAMLIT APP SOURCE
# ------------------------------
# Single-page Streamlit app for Single / Multi / Batch prediction modes.

streamlit_source = '''
import time, requests, pandas as pd, streamlit as st

BACKEND_URL = f"https://{HF_OWNER.lower()}-hill-country-revenue-forecast-backend.hf.space"

st.set_page_config(page_title="Hill Country Grocer Revenue Forecast", page_icon="🛒", layout="wide")

@st.cache_data(ttl=3600)
def fetch_info():
    r = requests.get(f"{BACKEND_URL}/info", timeout=30)
    r.raise_for_status()
    return r.json()

@st.cache_data(ttl=3600)
def fetch_sample():
    r = requests.get(f"{BACKEND_URL}/training_sample", timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["sample"])

def predict_one(f):
    r = requests.post(f"{BACKEND_URL}/predict", json={"features": f}, timeout=30)
    r.raise_for_status(); return r.json()

def predict_batch(records):
    r = requests.post(f"{BACKEND_URL}/predict_batch", json={"records": records}, timeout=60)
    r.raise_for_status(); return r.json()

@st.cache_data(show_spinner=False)
def sensitivity(feat_tuple, backend_url):
    features = dict(feat_tuple)
    base = predict_one(features)["prediction"]
    levers = []
    list_p = features.get("List_Price_USD", 5.0)
    promo_p = features.get("Promo_Price_USD", list_p)
    if abs(promo_p - list_p) < 0.01:
        r = predict_one({**features, "Promo_Price_USD": round(list_p * 0.9, 2)})
        levers.append((f"Promo at ${list_p*0.9:.2f} (10% off)", r["prediction"] - base))
    shelf = features.get("Shelf_Facings", 4)
    if shelf < 8:
        r = predict_one({**features, "Shelf_Facings": min(shelf + 4, 8)})
        levers.append((f"Shelf facings {shelf} → {min(shelf+4,8)}", r["prediction"] - base))
    levers.sort(key=lambda t: -t[1])
    return [(lab, d) for lab, d in levers if d > 0][:3]

st.title("Hill Country Grocer — Weekly Revenue Forecast")
st.caption("Split-conformal 95% prediction intervals · Powered by EvagAIML")

try:
    info = fetch_info()
    sample_df = fetch_sample()
except Exception as e:
    st.error(f"Backend unavailable: {e}"); st.stop()

fmap = {f["name"]: f for f in info["features"]}
mode = st.segmented_control("Mode", ["Single", "Multi", "Batch"], default="Single")

def store_fields(prefix, sample_df, fmap):
    src = st.radio("Store source", ["Existing store", "New store"], key=f"{prefix}_src", horizontal=True)
    sf = {}
    c1, c2 = st.columns(2)
    if src == "Existing store":
        opts = fmap["Store_Number"]["allowed_values"]
        sn = c1.selectbox("Store", opts, key=f"{prefix}_sn")
        row = sample_df[sample_df["Store_Number"] == sn]
        row = row.iloc[0] if len(row) else sample_df.iloc[0]
        sf = {"Store_Number": sn, "Store_Banner": str(row.get("Store_Banner","Hill Country Grocer")),
              "Store_Region": str(row.get("Store_Region","South - Texas Central")),
              "Store_Sq_Ft": float(row.get("Store_Sq_Ft", 42000)),
              "Store_Open_Year": int(row.get("Store_Open_Year", 2011))}
        c2.caption(f"Banner: {sf['Store_Banner']} · Region: {sf['Store_Region']}")
    else:
        sf["Store_Banner"] = c1.selectbox("Banner", fmap["Store_Banner"]["allowed_values"], key=f"{prefix}_banner")
        sf["Store_Region"] = c2.selectbox("Region", fmap["Store_Region"]["allowed_values"], key=f"{prefix}_region")
        sf["Store_Number"] = "HCG-101"
        n1, n2 = st.columns(2)
        sf["Store_Sq_Ft"] = float(n1.number_input("Sq ft", 5000, 100000, 42000, 1000, key=f"{prefix}_sqft"))
        sf["Store_Open_Year"] = int(n2.number_input("Year opened", 1990, 2025, 2011, key=f"{prefix}_yr"))
    return sf

def item_fields(prefix, fmap):
    ic1, ic2 = st.columns(2)
    dept = ic1.selectbox("Department", fmap["Department"]["allowed_values"], key=f"{prefix}_dept")
    brand = ic2.selectbox("Brand type", fmap["Brand_Type"]["allowed_values"], key=f"{prefix}_brand")
    n1, n2, n3 = st.columns(3)
    wt = float(n1.number_input("Weight (oz)", 0.1, 500.0, 16.0, 0.5, key=f"{prefix}_wt"))
    pack = float(n2.number_input("Pack size", 1, 24, 1, 1, key=f"{prefix}_pack"))
    facings = float(n3.number_input("Shelf facings", 1, 8, 4, 1, key=f"{prefix}_facings"))
    p1, p2 = st.columns(2)
    list_p = float(p1.number_input("List price ($)", 0.50, 50.0, 3.99, 0.10, key=f"{prefix}_list"))
    on_promo = p2.toggle("On promo", key=f"{prefix}_promo")
    promo_p = float(st.number_input("Promo price ($)", 0.25, list_p, round(list_p*0.9,2), 0.10, key=f"{prefix}_pp")) if on_promo else list_p
    return {"Department": dept, "Brand_Type": brand, "Net_Weight_oz": wt,
            "Pack_Size": pack, "Shelf_Facings": facings,
            "List_Price_USD": list_p, "Promo_Price_USD": promo_p}

if mode == "Single":
    store = store_fields("s", sample_df, fmap)
    item = item_fields("si", fmap)
    features = {**store, **item}
    if st.button("Predict", type="primary"):
        try:
            res = predict_one(features)
            p, lo, hi = res["prediction"], res["ci_low"], res["ci_high"]
            hw = (hi - lo) / 2
            st.success(f"**Predicted weekly revenue: ${p:,.2f} ± ${hw:,.2f}**")
            st.caption(f"95% PI: ${lo:,.2f} – ${hi:,.2f}")
            with st.status("Sensitivity analysis...") as s:
                levers = sensitivity(tuple(sorted(features.items())), BACKEND_URL)
                s.update(label="Done", state="complete")
            if levers:
                st.markdown("**Top levers:**")
                for lab, d in levers: st.markdown(f"- {lab}: **+${d:,.2f}/week**")
        except Exception as e:
            st.error(str(e))

elif mode == "Multi":
    n = int(st.number_input("Items in basket", 1, 10, 2))
    store = store_fields("m", sample_df, fmap)
    items = []
    for i in range(n):
        with st.expander(f"Item {i+1}", expanded=True):
            items.append(item_fields(f"mi{i}", fmap))
    records = [{**store, **it} for it in items]
    c1, c2 = st.columns(2)
    if c1.button("Predict basket", type="primary"):
        try:
            res = predict_batch(records)
            preds = res["predictions"]
            total = sum(p["prediction"] for p in preds)
            st.success(f"**Basket total: ${total:,.2f}/week**")
            rows = [{"Item": i+1, "Dept": records[i]["Department"],
                     "Revenue": f"${p['prediction']:,.2f}", "CI±": f"${(p['ci_high']-p['ci_low'])/2:,.2f}"}
                    for i, p in enumerate(preds)]
            st.dataframe(pd.DataFrame(rows), hide_index=True)
        except Exception as e:
            st.error(str(e))
    if c2.button("Across all stores"):
        try:
            stores = fmap["Store_Number"]["allowed_values"]
            all_recs = []
            for sn in stores:
                row = sample_df[sample_df["Store_Number"]==sn]
                row = row.iloc[0] if len(row) else sample_df.iloc[0]
                for it in items:
                    all_recs.append({**{"Store_Number":sn,"Store_Banner":str(row.get("Store_Banner","Hill Country Grocer")),
                                        "Store_Region":str(row.get("Store_Region","South - Texas Central")),
                                        "Store_Sq_Ft":float(row.get("Store_Sq_Ft",42000)),"Store_Open_Year":int(row.get("Store_Open_Year",2011))},**it})
            res = predict_batch(all_recs)
            n_items = len(items)
            totals = {sn: sum(p["prediction"] for p in res["predictions"][j*n_items:(j+1)*n_items]) for j,sn in enumerate(stores)}
            df_s = pd.DataFrame(sorted([{"Store":s,"Predicted total":f"${v:,.2f}"} for s,v in totals.items()],key=lambda r:-float(r["Predicted total"].replace("$","").replace(",",""))))
            st.dataframe(df_s, hide_index=True)
        except Exception as e:
            st.error(str(e))

elif mode == "Batch":
    req_cols = ["Department","Brand_Type","Net_Weight_oz","Pack_Size","List_Price_USD",
                "Promo_Price_USD","Shelf_Facings","Store_Number","Store_Banner","Store_Region","Store_Sq_Ft","Store_Open_Year"]
    up = st.file_uploader("Upload CSV", type=["csv"])
    if up:
        df_up = pd.read_csv(up)
        missing = [c for c in req_cols if c not in df_up.columns]
        if missing: st.warning(f"Missing columns: {missing}")
        else: st.success(f"{len(req_cols)}/{len(req_cols)} required columns present.")
        st.dataframe(df_up.head(5), hide_index=True)
        if not missing and st.button("Predict batch", type="primary"):
            try:
                res = predict_batch(df_up[req_cols].to_dict(orient="records"))
                preds = res["predictions"]
                df_out = df_up.copy()
                df_out["predicted_weekly_revenue_usd"] = [p["prediction"] for p in preds]
                df_out["ci_low"] = [p["ci_low"] for p in preds]
                df_out["ci_high"] = [p["ci_high"] for p in preds]
                st.dataframe(df_out, hide_index=True)
                st.markdown(f"**Total: ${df_out['predicted_weekly_revenue_usd'].sum():,.2f}**")
                st.download_button("Download CSV", df_out.to_csv(index=False).encode(),
                                   "hill_country_predictions.csv", "text/csv")
            except Exception as e:
                st.error(str(e))

st.divider()
st.caption(f"Model: {info.get('model_family','')} · R²={info.get('test_r2',0):.4f} · MAPE={info.get('test_mape_pct',0):.1f}% · 95% split-conformal PI")
'''

import os as _os
_base2 = pathlib.Path("/content") if _os.path.exists("/content") else pathlib.Path("/tmp")
(_base2 / "hf-frontend" / "src" / "streamlit_app.py").write_text(streamlit_source)
print("streamlit_app.py written.")


### Deploy to Hugging Face Spaces

**Process:** Create both HF Spaces with `space_sdk="docker"` and upload all required files. `exist_ok=True` means re-running this cell updates the Space rather than failing.

**Outcome:** Both Spaces are building on HF infrastructure. Build typically takes 3–7 minutes.

In [ ]:
# ------------------------------
# DEPLOY TO HUGGING FACE SPACES
# ------------------------------
# Creates both Spaces and uploads all required files.
# exist_ok=True: idempotent — re-run updates rather than errors.
import pathlib, os

# Colab uses /content; local execution falls back to /tmp
_is_colab = os.path.exists("/content")
base = pathlib.Path("/content") if _is_colab else pathlib.Path("/tmp")

# Backend Space
print(f"Creating backend Space: {BACKEND_REPO}")
api.create_repo(repo_id=BACKEND_REPO, repo_type="space", space_sdk="docker",
                exist_ok=True, private=False)

backend_files = [
    (str(base / "hf-backend" / "Dockerfile"), "Dockerfile"),
    (str(base / "hf-backend" / "requirements.txt"), "requirements.txt"),
    (str(base / "hf-backend" / "app.py"), "app.py"),
    (str(base / "model.joblib"), "model.joblib"),
    (str(base / "training_sample.parquet"), "training_sample.parquet"),
]
for local, remote in backend_files:
    print(f"  Uploading {remote}...")
    api.upload_file(path_or_fileobj=local, path_in_repo=remote,
                    repo_id=BACKEND_REPO, repo_type="space")

print(f"Backend uploaded → https://huggingface.co/spaces/{BACKEND_REPO}")

# Frontend Space
print(f"\nCreating frontend Space: {FRONTEND_REPO}")
api.create_repo(repo_id=FRONTEND_REPO, repo_type="space", space_sdk="docker",
                exist_ok=True, private=False)

frontend_files = [
    (str(base / "hf-frontend" / "Dockerfile"), "Dockerfile"),
    (str(base / "hf-frontend" / "requirements.txt"), "requirements.txt"),
    (str(base / "hf-frontend" / "src" / "streamlit_app.py"), "src/streamlit_app.py"),
]
for local, remote in frontend_files:
    print(f"  Uploading {remote}...")
    api.upload_file(path_or_fileobj=local, path_in_repo=remote,
                    repo_id=FRONTEND_REPO, repo_type="space")

print(f"Frontend uploaded → https://huggingface.co/spaces/{FRONTEND_REPO}")
print("\nBoth Spaces are building. Build typically takes 3–7 minutes.")

### Endpoint Smoke Test

**Process:** Wait for the backend to become available, then call each endpoint to verify responses. The `/predict` call uses a real feature dict. Includes the conformal coverage sanity check on a 100-row test sample.

**Analysis:** Empirical coverage should be between 0.85 and 1.00. If it falls outside that range, the conformal calibration may have been computed on a non-representative split or the model distribution has changed significantly between train and deploy environments.

**Outcome:** All five endpoints verified. Empirical coverage reported.

In [ ]:
# ------------------------------
# ENDPOINT SMOKE TEST AND COVERAGE CHECK
# ------------------------------
# Waits for backend readiness then verifies all 5 endpoints.
# Conformal coverage sanity check on a 100-row test sample.
import time, requests

print("Waiting for backend to start (up to 10 min)...")
for attempt in range(120):
    try:
        r = requests.get(f"{BACKEND_URL}/health", timeout=10)
        if r.status_code == 200:
            print(f"Backend live after ~{attempt * 5}s")
            break
    except Exception:
        pass
    if attempt % 12 == 0 and attempt > 0:
        print(f"  Still waiting... {attempt * 5}s elapsed")
    time.sleep(5)

# /health
print("\n/health:", requests.get(f"{BACKEND_URL}/health").json())

# /info
info_r = requests.get(f"{BACKEND_URL}/info").json()
print("/info ci_mechanism:", info_r["ci_mechanism"])
print("/info ci_calibration_n:", info_r["ci_calibration_n"])
print("/info model_family:", info_r["model_family"])

# /predict
sample_features = {
    "Department": "Produce", "Brand_Type": "National Brand",
    "Net_Weight_oz": 16.0, "Pack_Size": 1, "List_Price_USD": 1.99,
    "Promo_Price_USD": 1.75, "Shelf_Facings": 6,
    "Store_Number": "HCG-101", "Store_Banner": "Hill Country Grocer",
    "Store_Region": "South - Texas Central", "Store_Sq_Ft": 42000, "Store_Open_Year": 2011,
}
pred_r = requests.post(f"{BACKEND_URL}/predict", json={"features": sample_features}).json()
print(f"\n/predict: {pred_r}")

# /predict_batch (100 test rows)
test_records = X_test.iloc[:100].to_dict(orient="records")
batch_r = requests.post(f"{BACKEND_URL}/predict_batch", json={"records": test_records}).json()
print(f"/predict_batch: {batch_r['count']} predictions returned")

# Empirical coverage
y_test_100 = np.array(y_test.iloc[:100])
ci_lows  = np.array([p["ci_low"]  for p in batch_r["predictions"]])
ci_highs = np.array([p["ci_high"] for p in batch_r["predictions"]])
coverage = float(((y_test_100 >= ci_lows) & (y_test_100 <= ci_highs)).mean())
print(f"Empirical coverage on 100 test rows: {coverage:.3f} (target ~0.95)")
if 0.85 <= coverage <= 1.00:
    print("Coverage check: PASS")
else:
    print(f"Coverage check: FLAG — outside [0.85, 1.00]")

# /training_sample
ts_r = requests.get(f"{BACKEND_URL}/training_sample").json()
print(f"/training_sample: {ts_r['n']} rows")

# Frontend
fe_r = requests.get(f"https://{HF_OWNER.lower()}-hill-country-revenue-forecast-frontend.hf.space", timeout=30)
print(f"Frontend HTTP: {fe_r.status_code}")

## **Expanded Executive Summary**

### TLDR

**HistGradientBoostingRegressor** (max_depth=8) is the strongest model across the four-candidate sweep, reaching test R² = 0.8996 and MAE = $12.45 per item-store-week. Gradient boosting methods outperformed both bagging ensembles by 4–5 R² points on validation, consistent with the structured pricing and product hierarchy features in this dataset. The split-conformal 95% prediction interval (±$40.98 average half-width) is model-agnostic and calibrated on 1,776 held-out rows, meaning it provides valid coverage regardless of which model the harness picks on future retrains.

### Full Summary

**Objective:** Build and deploy a weekly revenue forecast model for the Hill Country Grocer chain that predicts `Weekly_Revenue_USD` per item × store combination with a calibrated 95% prediction interval. The deployed model supports merchandising decisions including promo pricing, shelf allocation, and ordering quantities, giving managers a quantitative baseline before the week starts.

**Iterative Development:** The gated pipeline harness swept four candidate algorithms. HistGradientBoosting and XGBoost both outperformed the ensemble methods by val R² margin, confirming that additive residual learning adds meaningful signal on this structured dataset. GridSearchCV over `max_depth ∈ [4, 6, 8]` selected depth 8, which captures item × store × department interactions at sufficient depth without overfitting the training set. The split-conformal CI mechanism was chosen over tree-level variance or native XGBoost uncertainty because it is model-agnostic and calibrated — the coverage guarantee holds for any harness winner, not just RandomForest.

**Performance Analysis:** Test R² = 0.8996 explains approximately 90% of weekly revenue variance in the unseen holdout. MAPE = 15.4% is higher than the R² would suggest, driven by the long tail of low-price, low-volume produce and bakery items where even small absolute errors produce large percentage errors. MAE = $12.45/week is the operationally relevant metric: for most items in the $5–$20 weekly revenue range, this places the forecast within 10–15% of the actual. Top predictors by permutation importance:
  1. `List_Price_USD`: 1.35
  2. `Promo_Price_USD`: 0.26
  3. `Net_Weight_oz`: 0.24
  4. `Shelf_Facings`: 0.15
  5. `Store_Region_South - Texas Hill Country`: 0.14

List price dominates, which makes physical sense: revenue = price × units sold, so the price ceiling is the primary determinant of the revenue range. Promo price is second, capturing the discount lever.

**Economic Impact:** At $12.45 MAE per item-store-week, a chain running 6 stores × 200 SKUs per week can expect total weekly forecast error in the range of $14,940. For over-order decisions costing roughly 10% of over-forecast value in markdowns and waste, accurate forecasting at this level avoids roughly $1,494/week in preventable markdown expense across the chain. The promo sensitivity feature (perturbing `Promo_Price_USD` by ±10%) enables category managers to see forecast ROI before committing to a flyer, directly reducing unnecessary discounts.

**Deployment Readiness:** Backend runs as a Docker container on Hugging Face Spaces (`EvagAIML/hill-country-revenue-forecast-backend`) exposing `/health`, `/info`, `/predict`, `/predict_batch`, and `/training_sample`. Frontend runs as a Docker + Streamlit container (`EvagAIML/hill-country-revenue-forecast-frontend`) with Single/Multi/Batch prediction modes. Enterprise integration path: call `/predict_batch` from the ERP or inventory system with the upcoming week's planned item-store combinations. SMB integration path: category managers use the Streamlit UI at the weekly planning meeting.

**Next Steps:** The most impactful improvements in rough priority order: (1) Add a week-of-year column to the dataset — the current model is time-blind, missing seasonal patterns like holiday lifts and summer slumps that likely account for a portion of the current MAPE. (2) Retrain on a rolling 52-week window as live sales data accumulates — the training set is a static snapshot; model drift is expected after 6–12 months. (3) Add store-level daily foot traffic or POS volume as an external signal — the current features are all item or store attributes; transaction volume is a more direct predictor of weekly revenue.